# arbital: a hands-on tutorial

**arbital** is a small Python package for exploring general association between variables. For a
chosen target, it measures every other variable with both a correlation-based statistic and
mutual information, and draws the result as orbits around the target: radius encodes strength,
orbit shape encodes non-monotone structure, angle groups mutually associated variables, and an
optional feature-selection pass sizes the markers.

This notebook works through the package feature by feature. It uses only NumPy and the optional
bundled example data. By Aaron Byrne, PhD.


## Setup

The core package needs only NumPy. `plotly` is used here to render figures inline; the library
itself can write standalone HTML without it.


In [ ]:
import sys, pathlib
src = pathlib.Path.cwd().parent / "src"      # adjust if pip-installed
if src.exists():
    sys.path.insert(0, str(src))

import numpy as np
import arbital
from arbital import datasets
print("arbital", arbital.__version__)   # datasets load via seaborn (optional extra)

## 1. A first figure

`orbits()` accepts a 2-D array, a DataFrame, or a bundled `Table`, plus a target column. Marker
size is the relative feature-selection gain by default, so the largest markers are the strongest
non-redundant variables.


In [ ]:
cars = datasets.load_mpg()
print(cars, "|", cars.notes)

sys = arbital.orbits(cars, target="mpg")
sys.to_plotly()      # or sys.to_html("mpg.html") to open in a browser

In [ ]:
for r in sys.table():
    print(f"{r['name']:<13} r_I={r['r_info']:.2f}  nu={r['nonlinearity']:.0%}  "
          f"pick={r['pick']}  gain={r['gain']:+.2f}")

## 2. What the measures see

Different measures detect different departures from independence. Compare Pearson (linear),
Spearman (monotone), and mutual information (any dependence) on four relationships.


In [ ]:
rng = np.random.default_rng(0)
x = rng.standard_normal(800)
cases = {"linear": x + 0.3*rng.standard_normal(800),
         "monotone (exp)": np.exp(x),
         "parabola": x**2 + 0.1*rng.standard_normal(800),
         "sine": np.sin(3*x)}
print(f"{'relationship':<15}{'Pearson':>9}{'Spearman':>10}{'MI(nats)':>10}{'r_I':>7}")
for name, y in cases.items():
    mi = arbital.mutual_information(x, y)
    print(f"{name:<15}{arbital.pearson(x,y):>9.2f}{arbital.spearman(x,y):>10.2f}"
          f"{mi:>10.2f}{arbital.linfoot(mi):>7.2f}")

Pearson misses the exponential curve, Spearman recovers it, and both miss the
parabola and sine, where only mutual information stays high. That gap is what the ghost marker
and tether in the figure represent.

## 3. Transformed variables

When several monotone transformations of one quantity are present (a raw value, its log, sqrt, a
capped version), they are largely redundant. They cluster at nearby angles, and the selection
keeps the one that best matches the target.


In [ ]:
base = rng.uniform(1, 60, 800)
cols = {"y": np.log(base) + 0.1*rng.standard_normal(800),
        "raw": base, "log": np.log(base),
        "sqrt": np.sqrt(base), "capped": np.minimum(base, 30)}
X = np.column_stack([cols[c] for c in cols])

class Frame:                     # minimal DataFrame-like wrapper
    columns = list(cols)
    def __array__(self, dtype=None): return X

for p in arbital.select_features(Frame(), target="y"):
    print(f"{p['pick']}. {p['name']:<8} gain={p['gain']:+.2f}")
arbital.orbits(Frame(), target="y").to_plotly()

## 4. Lagged variables and time dependence

Given a series and its own lags, the orbits show how far back temporal dependence reaches. Here
is a first-order autoregressive series.


In [ ]:
n = 600
s = np.zeros(n)
for t in range(1, n):
    s[t] = 0.8*s[t-1] + rng.standard_normal()
lags = {"series_t": s, "lag_1": np.roll(s,1), "lag_2": np.roll(s,2),
        "lag_3": np.roll(s,3), "lag_8": np.roll(s,8)}
data = np.column_stack([lags[c][10:] for c in lags])

class LagFrame:
    columns = list(lags)
    def __array__(self, dtype=None): return data

lsys = arbital.orbits(LagFrame(), target="series_t")
for r in lsys.table():
    print(f"{r['name']:<9} r_I={r['r_info']:.2f}  pick={r['pick']}")
lsys.to_plotly()

## 5. Categorical and mixed data

If the target or a feature is categorical, arbital selects the appropriate mutual-information
estimator automatically. String columns in a bundled Table are detected; others can be passed
via `categorical=`.


In [ ]:
tit = datasets.load_titanic()
print(tit, "| categorical:", tit.categorical)
tsys = arbital.orbits(tit, target="survived")
for r in tsys.table():
    tag = "categorical" if r["categorical"] else ""
    print(f"{r['name']:<10} r_I={r['r_info']:.2f}  r={r['pearson']:+.2f}  {tag}")
tsys.to_plotly()

## 6. Feature selection on its own

`select_features` returns the greedy mRMR ranking with no figure. Redundancy is charged only
against variables already selected.


In [ ]:
for row in arbital.select_features(cars, target="mpg"):
    note = "" if row["gain"] > 0 else "   (redundant)"
    print(f"{row['pick']}. {row['name']:<13} gain={row['gain']:+.2f}{note}")

Pass `selection=True` to label markers with pick numbers in the figure:

In [ ]:
arbital.orbits(cars, target="mpg", selection=True).to_plotly()

## 7. Estimation uncertainty

By default the arc length is fixed and only its curvature (the nonlinear share) is meaningful.
`uncertainty=True` computes a bootstrap standard error of r_I and maps it to the arc's angular
extent, so a wider arc indicates a less certain estimate.


In [ ]:
usys = arbital.orbits(datasets.load_penguins(), target="body_mass_g",
                      uncertainty=True, n_bootstrap=200)
for r in usys.table():
    print(f"{r['name']:<18} r_I={r['r_info']:.2f} +/- {r['r_info_se']:.03f}")
usys.to_plotly()

## 8. Your own data

Any array or DataFrame-like object works. Useful parameters to explore:

- `scale="linear"` instead of the default log-in-information radial scale
- `angle_layout="embed"` so angular gaps reflect the association structure
- `size="rinfo"` or `size="uniform"` to change what marker size means
- `target=None` to auto-select the most central variable

Every estimator is a short, readable NumPy function in `arbital/measures.py`.


In [ ]:
rng = np.random.default_rng(1)
t = rng.standard_normal(600)
mine = np.column_stack([t, t + 0.3*rng.standard_normal(600),
                        np.sin(2*t) + 0.2*rng.standard_normal(600),
                        rng.standard_normal(600)])
arbital.orbits(mine, target=0).to_plotly()